# Detección Inteligente de Anomalías en Tráfico de Red

## Introducción

La detección automática de anomalías es fundamental en sistemas de seguridad modernos. 
Este cuaderno implementa dos enfoques complementarios:

1. **Machine Learning Clásico** — Isolation Forest para detección no supervisada
2. **Deep Learning** — Autoencoder neuronal para modelado de normalidad

Ambos métodos identifican patrones de tráfico inusuales que pueden indicar:
- Intentos de exfiltración de datos
- Conexiones a servidores maliciosos
- Comportamientos anómalos de sistemas
- Amplificación de ataques (botnet activity)

## 1. Importaciones y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

# Reproducibilidad
SEED = 42
np.random.seed(SEED)

# Estilo de gráficos
sns.set_theme(style="whitegrid", palette="muted")
print("Importaciones OK")

## 2. Generación del dataset sintético de tráfico de red

Simulamos un dataset con 1 000 conexiones normales y 50 anómalas.
Las conexiones anómalas tienen volúmenes de datos y duraciones fuera
del rango habitual.

In [ ]:
def generar_trafico_red(n_normal=1000, n_anomalias=50, seed=42):
    """
    Genera un DataFrame simulando tráfico de red.
    Columnas: bytes_sent, bytes_recv, duration, src_port, dst_port, protocol
    label: 0=normal, 1=anomalía (solo para evaluación, no se usa en entrenamiento)
    """
    rng = np.random.default_rng(seed)

    # Tráfico normal — distribuciones realistas
    normal = pd.DataFrame({
        "bytes_sent":  rng.exponential(scale=5_000,  size=n_normal),
        "bytes_recv":  rng.exponential(scale=15_000, size=n_normal),
        "duration":    rng.exponential(scale=30,     size=n_normal),
        "src_port":    rng.integers(1024, 65535,      size=n_normal),
        "dst_port":    rng.choice([80, 443, 22, 8080, 3306], size=n_normal),
        "protocol":    rng.choice([6, 17],             size=n_normal),  # TCP=6, UDP=17
        "label":       0
    })

    # Anomalías — exfiltración masiva de datos o conexiones largas
    anomalias = pd.DataFrame({
        "bytes_sent":  rng.exponential(scale=500_000, size=n_anomalias),
        "bytes_recv":  rng.exponential(scale=2_000,   size=n_anomalias),
        "duration":    rng.exponential(scale=600,     size=n_anomalias),
        "src_port":    rng.integers(1024, 65535,       size=n_anomalias),
        "dst_port":    rng.integers(1, 1023,           size=n_anomalias),
        "protocol":    rng.choice([6, 17],              size=n_anomalias),
        "label":       1
    })

    df = pd.concat([normal, anomalias], ignore_index=True).sample(
        frac=1, random_state=seed
    )
    return df

df = generar_trafico_red()
print(f"Dataset: {df.shape}  |  Normal: {(df.label==0).sum()}  |  Anomalías: {(df.label==1).sum()}")
df.head()

## 3. Preprocesamiento y normalización

Escalamos todas las características al rango **[0, 1]** usando `MinMaxScaler`.
Este paso es crucial para que el algoritmo no se vea afectado por la escala de
las variables.

In [ ]:
FEATURE_COLS = ["bytes_sent", "bytes_recv", "duration", "src_port", "dst_port", "protocol"]

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(df[FEATURE_COLS])
y_true   = df["label"].values  # solo para evaluación

print(f"Rango antes  — bytes_sent: [{df['bytes_sent'].min():.0f}, {df['bytes_sent'].max():.0f}]")
print(f"Rango después— bytes_sent: [{X_scaled[:,0].min():.4f}, {X_scaled[:,0].max():.4f}]")

# Visualizar distribución original
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(["bytes_sent", "duration", "bytes_recv"]):
    axes[i].hist(df[col], bins=50, color="steelblue", alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel("Valor")
plt.suptitle("Distribución de características de tráfico de red", y=1.02)
plt.tight_layout()
plt.show()

## 4. Isolation Forest

El **Isolation Forest** crea particiones aleatorias recursivas.
Los puntos que se aíslan rápidamente (con pocas particiones) son considerados _anómalos_.

- `contamination`: fracción esperada de anomalías en el dataset.
- `n_estimators`: número de árboles de aislamiento.
- Predicción: **+1** = normal, **-1** = anomalía.

In [ ]:
# Entrenamos con TODOS los datos (aprendizaje no supervisado)
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # esperamos ~5% de anomalías
    random_state=SEED,
    n_jobs=-1
)
iso_forest.fit(X_scaled)

# Predicción y puntuación
pred_iso   = iso_forest.predict(X_scaled)          # +1 normal / -1 anomalía
scores_iso = iso_forest.decision_function(X_scaled) # negativo = más anómalo

df_result = df.copy()
df_result["pred_iso"]   = pred_iso
df_result["score_iso"]  = scores_iso

n_detectadas = (pred_iso == -1).sum()
print(f"Anomalías detectadas: {n_detectadas} de {len(pred_iso)}")
print()

# Convertir predicción a etiqueta 0/1 para comparar con la verdad
pred_bin = (pred_iso == -1).astype(int)
print("=== Reporte de clasificación ===")
print(classification_report(y_true, pred_bin, target_names=["Normal", "Anomalía"]))

In [ ]:
# ── Visualización: bytes_sent vs duration coloreado por predicción ──────
colors = np.where(pred_iso == -1, "red", "steelblue")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de dispersión
axes[0].scatter(df["bytes_sent"], df["duration"],
                c=colors, alpha=0.5, s=15)
axes[0].set_xlabel("bytes_sent")
axes[0].set_ylabel("duration (s)")
axes[0].set_title("Isolation Forest — Detección de Anomalías")
axes[0].legend(
    handles=[
        plt.scatter([], [], c="steelblue", label="Normal",   s=30),
        plt.scatter([], [], c="red",       label="Anomalía", s=30)
    ]
)

# Histograma de puntuaciones
axes[1].hist(scores_iso[pred_iso ==  1], bins=40, alpha=0.6, color="steelblue", label="Normal")
axes[1].hist(scores_iso[pred_iso == -1], bins=40, alpha=0.8, color="red",       label="Anomalía")
axes[1].set_xlabel("Puntuación de anomalía")
axes[1].set_ylabel("Frecuencia")
axes[1].set_title("Distribución de puntuaciones")
axes[1].legend()

plt.tight_layout()
plt.show()

# Top 5 anomalías más extremas
print("\nTop 5 anomalías más extremas:")
print(df_result[df_result.pred_iso == -1]
      .sort_values("score_iso")[FEATURE_COLS + ["score_iso"]].head())

## 5. Autoencoder para detección de anomalías

Un **autoencoder** aprende a comprimir y reconstruir datos normales.
Cuando procesa un dato anómalo, el **error de reconstrucción (MSE)** es
significativamente mayor, lo que permite detectar intrusiones no vistas
durante el entrenamiento.

No necesita TensorFlow — implementamos uno minimalista con `numpy`.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

# ── Separar datos normales para entrenamiento ───────────────────────────
X_normal = X_scaled[y_true == 0]   # solo tráfico normal
X_all    = X_scaled                 # todos para inferencia

X_train_ae, X_val_ae = train_test_split(X_normal, test_size=0.15, random_state=SEED)

# ── Autoencoder con MLPRegressor (entrada = salida) ─────────────────────
# Arquitectura: 6 → 8 → 3 → 8 → 6
autoencoder = MLPRegressor(
    hidden_layer_sizes=(8, 3, 8),
    activation="relu",
    solver="adam",
    max_iter=300,
    random_state=SEED,
    early_stopping=True,
    validation_fraction=0.15
)
# Entrenamos para reconstruir la entrada (X → X)
autoencoder.fit(X_train_ae, X_train_ae)

print(f"Épocas de entrenamiento: {autoencoder.n_iter_}")
best_loss = getattr(autoencoder, "best_loss_", None) or autoencoder.loss_curve_[-1]
print(f"Pérdida final: {best_loss:.6f}")

In [ ]:
# ── Calcular error de reconstrucción para todos los puntos ──────────────
X_reconstructed = autoencoder.predict(X_all)
mse_por_muestra = np.mean(np.power(X_all - X_reconstructed, 2), axis=1)

# Umbral: percentil 95 del error en datos de validación
X_val_reconstructed = autoencoder.predict(X_val_ae)
mse_val    = np.mean(np.power(X_val_ae - X_val_reconstructed, 2), axis=1)
umbral_ae  = np.percentile(mse_val, 95)
print(f"Umbral de detección (P95 validación): {umbral_ae:.6f}")

# Clasificar
pred_ae  = (mse_por_muestra > umbral_ae).astype(int)
print(f"Anomalías detectadas: {pred_ae.sum()}")
print()
print("=== Reporte de clasificación (Autoencoder) ===")
print(classification_report(y_true, pred_ae, target_names=["Normal", "Anomalía"]))

# ── Visualización del error de reconstrucción ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mse_por_muestra, lw=0.5, color="steelblue", label="MSE reconstrucción")
ax.axhline(umbral_ae, color="red", ls="--", lw=1.5, label=f"Umbral = {umbral_ae:.4f}")
idx_anomalias = np.where(y_true == 1)[0]
ax.scatter(idx_anomalias, mse_por_muestra[idx_anomalias],
           color="red", s=25, zorder=5, label="Anomalía real")
ax.set_xlabel("Índice de muestra")
ax.set_ylabel("MSE")
ax.set_title("Error de reconstrucción del Autoencoder")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Comparación de ambos modelos

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

modelos = {
    "Isolation Forest": pred_bin,
    "Autoencoder (MLP)": pred_ae,
}

print(f"{'Modelo':<22} {'Precisión':>10} {'Recall':>8} {'F1':>8}")
print("-" * 52)
for nombre, pred in modelos.items():
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred)
    f = f1_score(y_true, pred)
    print(f"{nombre:<22} {p:>10.4f} {r:>8.4f} {f:>8.4f}")

print("\n✓ Cuaderno 2 completado.")